# Phase 1 — Data Prep & Baseline (Module 2)

**Project:** Automated Code Review AI with Vulnerability & Style Feedback
**This notebook (self-contained, run top-to-bottom on Colab with a GPU runtime):**

1. Download the **CodeReviewer** *comment-generation* (`msg`) dataset from Zenodo.
2. Build splits: **Python/JS** for evaluation, full **multilingual** train pool.
3. Load **CodeT5+ (`Salesforce/codet5p-220m`)** and run **baseline inference**
   (pre-fine-tuning) on a handful of real diffs — this is the "before" half of the
   before/after comparison the report needs.

> **Runtime:** set `Runtime -> Change runtime type -> GPU` before running.
> **Proof-of-work:** run every cell, then **commit this notebook with outputs intact**
> to `ml/notebooks/` — do NOT clear outputs.

Base paper: *CodeReviewer: Pre-Training for Automating Code Review Activities*
(arXiv:2203.09095). Data: Zenodo record 6900648.


## 0. Environment


In [14]:
# Pin transformers to 4.44.2: newer (4.5x / 5.x, which Colab now ships) breaks the
# CodeT5+ tokenizer ("Input must be a List[Union[str, AddedToken]]"). If this cell
# DOWNGRADES transformers, do Runtime -> Restart session, then Run all again.
!pip -q install "transformers==4.44.2" "datasets>=2.20" "accelerate>=0.33" \
                "evaluate>=0.4" "sacrebleu>=2.4"
import torch, transformers, platform
print("python     :", platform.python_version())
print("torch      :", torch.__version__)
print("transformers:", transformers.__version__)
print("CUDA avail :", torch.cuda.is_available(),
      "|", (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only"))


python     : 3.12.13
torch      : 2.11.0+cu128
transformers: 4.44.2
CUDA avail : True | Tesla T4


In [15]:
import os, json, zipfile, urllib.request, glob, random, textwrap
from pathlib import Path

# --- Config -----------------------------------------------------------------
MODEL_NAME = "Salesforce/codet5p-220m"   # smallest CodeT5+ encoder-decoder
LANGS      = {"py", "python", ".py", "js", "javascript", ".js"}  # matched case-insensitively
MAX_SRC    = 512    # input (diff) token budget
MAX_TGT    = 128    # generated comment token budget
SEED       = 42
random.seed(SEED)

# Persist prepared data + artifacts to Google Drive so a runtime disconnect
# doesn't lose the 846 MB download. Set USE_DRIVE = False to keep everything on
# the ephemeral Colab disk instead.
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/code-review-ai")
else:
    ROOT = Path("/content/code-review-ai")

RAW_DIR   = ROOT / "data" / "raw"
PREP_DIR  = ROOT / "data" / "prepared"
ART_DIR   = ROOT / "artifacts"
for d in (RAW_DIR, PREP_DIR, ART_DIR):
    d.mkdir(parents=True, exist_ok=True)
print("ROOT =", ROOT)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ROOT = /content/drive/MyDrive/code-review-ai


## 1. Download the CodeReviewer `msg` (comment-generation) data

`Comment_Generation.zip` is ~846 MB. Downloaded once, cached under `data/raw/`.
Records are JSONL; the training file's fields are `oldf`, `patch`, `msg`, `id`, `y`, while `valid`/`test` also carry `lang` and `proj`. We inspect the exact keys below rather than assume.


In [16]:
ZENODO_URL = "https://zenodo.org/records/6900648/files/Comment_Generation.zip?download=1"
ZIP_PATH   = RAW_DIR / "Comment_Generation.zip"

if not ZIP_PATH.exists():
    print("Downloading (~846 MB, one-time)...")
    urllib.request.urlretrieve(ZENODO_URL, ZIP_PATH)
    print("Saved ->", ZIP_PATH, f"({ZIP_PATH.stat().st_size/1e6:.1f} MB)")
else:
    print("Already downloaded ->", ZIP_PATH, f"({ZIP_PATH.stat().st_size/1e6:.1f} MB)")

EXTRACT_DIR = RAW_DIR / "Comment_Generation"
if not EXTRACT_DIR.exists():
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(RAW_DIR)
    print("Extracted.")

# Discover the actual .jsonl layout inside the archive (don't assume filenames).
jsonl_files = sorted(glob.glob(str(RAW_DIR / "**" / "*.jsonl"), recursive=True))
print(f"\nFound {len(jsonl_files)} .jsonl files:")
for f in jsonl_files:
    print("  ", os.path.relpath(f, RAW_DIR))


Already downloaded -> /content/drive/MyDrive/code-review-ai/data/raw/Comment_Generation.zip (846.6 MB)

Found 3 .jsonl files:
   Comment_Generation/msg-test.jsonl
   Comment_Generation/msg-train.jsonl
   Comment_Generation/msg-valid.jsonl


In [17]:
# Inspect one record so we know the exact field names + language encoding.
def read_jsonl(path, limit=None):
    rows = []
    with open(path, encoding="utf-8") as fh:
        for i, line in enumerate(fh):
            if limit is not None and i >= limit:
                break
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

sample = read_jsonl(jsonl_files[0], limit=1)[0]
print("Keys:", list(sample.keys()))
print()
for k, v in sample.items():
    preview = str(v).replace("\n", "\\n")
    print(f"- {k:12s}: {preview[:160]}")


Keys: ['patch', 'y', 'oldf', 'idx', 'id', 'msg', 'proj', 'lang']

- patch       : @@ -53,7 +53,7 @@ public class ProtocGapicPluginGeneratorTest {\n                 model.getFiles().stream().map(ProtoFile::getProto).collect(Collectors.toList()
- y           : 1
- oldf        : /* Copyright 2019 Google LLC\n *\n * Licensed under the Apache License, Version 2.0 (the "License");\n * you may not use this file except in compliance with the
- idx         : 1
- id          : 30879
- msg         : can we also test for `transport=rest`?
- proj        : googleapis-gapic-generator
- lang        : java


## 2. Build splits — evaluate on Python/JS, train on the multilingual pool

The `msg` release has a **schema quirk**: `valid`/`test` carry a `lang` field, but
`train` does **not** (its keys are just `oldf/patch/msg/id/y` — no `lang`, no `proj`).
So we can't filter train by language, and per-record language-guessing across 9
languages is fragile.

Decision: **filter valid/test to Python + JavaScript** (that's where we *measure* —
this is where the Python/JS scope and the "JS validation" gap live), and keep the
**full multilingual `train`** as the training pool (review-comment style transfers
across languages; 117k rows >> ~4k Python/JS rows). We print each file's real
language distribution so nothing is assumed.


In [18]:
# Map discovered files to splits by name.
def pick(split):
    hits = [f for f in jsonl_files if split in os.path.basename(f).lower()]
    return hits[0] if hits else None

split_files = {s: pick(s) for s in ("train", "valid", "test")}
# Some releases use 'dev' / 'eval' for the validation split — fall back if needed.
if split_files["valid"] is None:
    for alt in ("dev", "eval", "val"):
        p = pick(alt)
        if p:
            split_files["valid"] = p
            break
print("Split -> file:")
for s, f in split_files.items():
    print(f"  {s:6s}: {os.path.relpath(f, RAW_DIR) if f else 'MISSING'}")

# Detect the language field name from the sample keys.
LANG_KEY = next((k for k in sample if k.lower() in ("lang", "language")), None)
print("\nDetected language key:", LANG_KEY)


Split -> file:
  train : Comment_Generation/msg-train.jsonl
  valid : Comment_Generation/msg-valid.jsonl
  test  : Comment_Generation/msg-test.jsonl

Detected language key: lang


In [19]:
from collections import Counter

prepared = {}
for split, path in split_files.items():
    if not path:
        print(f"[skip] {split}: no file"); continue
    rows = read_jsonl(path)
    # A split is language-taggable only if 'lang' is actually populated.
    has_lang = any(str(r.get("lang", "")).strip() for r in rows[:200])
    if has_lang:
        dist = Counter(str(r.get("lang", "")).strip().lower() for r in rows)
        kept = [r for r in rows if str(r.get("lang", "")).strip().lower() in LANGS]
        out = PREP_DIR / f"{split}.pyjs.jsonl"
        print(f"{split}: {len(rows)} total | langs {dict(dist)}")
        print(f"  -> kept {len(kept)} Python/JS  ->  {out.name}")
    else:
        kept = rows                                   # multilingual training pool
        out = PREP_DIR / f"{split}.all.jsonl"
        print(f"{split}: {len(rows)} total | no 'lang' field -> keep all (multilingual)")
        print(f"  -> wrote all {len(kept)}  ->  {out.name}")
    with open(out, "w", encoding="utf-8") as fh:
        for r in kept:
            fh.write(json.dumps(r) + "\n")
    prepared[split] = kept
    print()

print("Prepared sizes:", {k: len(v) for k, v in prepared.items()},
      "\n(train = multilingual pool; valid/test = Python/JS for evaluation)")


train: 117739 total | no 'lang' field -> keep all (multilingual)
  -> wrote all 117739  ->  train.all.jsonl

valid: 10319 total | langs {'py': 1420, '.cs': 682, 'js': 1035, 'java': 1636, 'rb': 1049, 'php': 443, 'go': 2826, 'c': 492, 'cpp': 736}
  -> kept 2455 Python/JS  ->  valid.pyjs.jsonl

test: 10169 total | langs {'java': 1658, 'php': 426, 'rb': 1000, 'js': 967, 'cpp': 731, 'py': 1403, 'c': 453, 'go': 2840, '.cs': 691}
  -> kept 2370 Python/JS  ->  test.pyjs.jsonl

Prepared sizes: {'train': 117739, 'valid': 2455, 'test': 2370} 
(train = multilingual pool; valid/test = Python/JS for evaluation)


## 3. Load CodeT5+ baseline (`Salesforce/codet5p-220m`)


In [20]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)   # works on transformers 4.44.2
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device).eval()
n_params = sum(p.numel() for p in model.parameters())
print(f"Loaded {MODEL_NAME}  ({n_params/1e6:.0f}M params) on {device}")


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Loaded Salesforce/codet5p-220m  (223M params) on cuda


## 4. Baseline inference (pre-fine-tuning)

Feed the diff hunk, generate a review comment. The **pretrained** CodeT5+ has not
seen the review task, so expect generic/weak output — that's the point: this is the
"before" baseline we compare the fine-tuned model against in Phase 2.


In [21]:
DIFF_KEY = next((k for k in sample if k.lower() in
                 ("diff_hunk", "patch", "diff", "old_file")), "diff_hunk")
GOLD_KEY = next((k for k in sample if k.lower() in ("comment", "msg", "target")), "comment")
print("Using input field:", DIFF_KEY, "| gold comment field:", GOLD_KEY)

@torch.no_grad()
def review(diff_text, num_beams=4):
    enc = tokenizer(diff_text, truncation=True, max_length=MAX_SRC,
                    return_tensors="pt").to(device)
    out = model.generate(**enc, max_new_tokens=MAX_TGT, num_beams=num_beams)
    return tokenizer.decode(out[0], skip_special_tokens=True)


Using input field: patch | gold comment field: msg


In [22]:
eval_split = prepared.get("test") or prepared.get("valid") or prepared.get("train")
examples = random.sample(eval_split, k=min(5, len(eval_split)))

baseline_records = []
for i, ex in enumerate(examples, 1):
    diff = str(ex.get(DIFF_KEY, ""))
    gold = str(ex.get(GOLD_KEY, ""))
    pred = review(diff)
    baseline_records.append({"diff": diff, "gold_comment": gold, "baseline_pred": pred})
    print("=" * 90)
    print(f"[Example {i}] language={ex.get(LANG_KEY)}")
    print("--- DIFF (truncated) ---")
    print(textwrap.shorten(diff.replace("\n", " "), width=400))
    print("--- GOLD reviewer comment ---");     print(gold[:300])
    print("--- BASELINE CodeT5+ prediction ---"); print(pred[:300])
print("=" * 90)


[Example 1] language=py
--- DIFF (truncated) ---
@@ -1166,12 +1166,16 @@ def getControlFieldSpeech(attrs,ancestorAttrs,fieldType,formatConfig=None,extraD # speakStatesFirst: Speak the states before the role. speakStatesFirst=role==controlTypes.ROLE_LINK + containerContainsText="" #: used for item counts for lists + # Determine what text to speak. # Special cases if speakEntry and childControlCount and [...]
--- GOLD reviewer comment ---
Is it still important to have `speakEntry` here? It is checked before this value `containerContainsText` is used in the "General" section.
--- BASELINE CodeT5+ prediction ---
/*
 * Licensed to the Apache Software Foundation (ASF) under one or more
 * contributor license agreements.  See the NOTICE file distributed with
 * this work for additional information regarding copyright ownership.
 * The ASF licenses this file to You under the Apache License, Version 2.0
 * (the 
[Example 2] language=py
--- DIFF (truncated) ---
@@ -45,7 +45,7 @@ class FPN(nn.Mod

## 5. Save baseline examples (for the before/after report table)


In [23]:
ART_PATH = ART_DIR / "phase1_baseline_examples.json"
with open(ART_PATH, "w", encoding="utf-8") as fh:
    json.dump({
        "model": MODEL_NAME,
        "note": "Pre-fine-tuning baseline. Compare against Phase 2 fine-tuned outputs.",
        "split_sizes": {k: len(v) for k, v in prepared.items()},
        "examples": baseline_records,
    }, fh, indent=2)
print("Saved ->", ART_PATH)
print("\nSplit sizes (Python+JS):", {k: len(v) for k, v in prepared.items()})


Saved -> /content/drive/MyDrive/code-review-ai/artifacts/phase1_baseline_examples.json

Split sizes (Python+JS): {'train': 117739, 'valid': 2455, 'test': 2370}


## Next: Phase 2 — Fine-tuning

Phase 2 will:
train CodeT5+ on `data/prepared/train.all.jsonl`, evaluate **BLEU-4** and
**CodeBLEU** against the paper baseline (14.8) and this project's targets
(≥18 / ≥22), and export the fine-tuned weights.
